In [0]:
# 必要なライブラリのインストール
%pip install -U -qqqq databricks-sdk databricks-ai-search
dbutils.library.restartPython()

# 00-0. 代表者：共通の AI Search エンドポイントの作成

**このノートブックは代表者が1人だけ実行する。参加者は実行しない。**

AI Search のエンドポイントは、参加者全員で1つを共有する。代表者がエンドポイントを作り、参加者がそこに自分の索引を作れるよう権限を付ける。

| | 作る人 | ノートブック |
|---|---|---|
| エンドポイント（全員で1つ） | 代表者 | このノートブック |
| 索引（参加者ごと） | 参加者それぞれ | 1-1 |

## このノートブックでやること
1. パラメータの設定（widget）
2. Standard のエンドポイントを作る
3. 参加者に「使用可能」（CAN USE）の権限を付ける
4. 参加者の索引の数を確かめる（ハンズオン中）
5. 後片付け（ハンズオンの後）

## 前提
- ハンズオンの開始の **10分以上前** に実行する。エンドポイントの作成には数分かかる
- エンドポイントは **稼働している間ずっと課金される**。全員が終わったら、5 の後片付けで削除する
- 1つのエンドポイントに作れる索引は **50** までである。参加者が多い場合は、`AI_SEARCH_ENDPOINT` を変えてこのノートブックをもう一度実行し、2つ目のエンドポイントを作って参加者を振り分ける

---

## 0-0-1. パラメータの設定

| widget | 内容 |
|---|---|
| `AI_SEARCH_ENDPOINT` | 作る共通のエンドポイントの名前。参加者に伝え、1-1 の widget に同じ名前を入れてもらう |
| `PARTICIPANTS` | 権限を付ける相手。グループ名（例 `users` はワークスペースの全ユーザー）、またはユーザーのメールアドレス。カンマ区切りで複数指定できる |

In [0]:
dbutils.widgets.text("AI_SEARCH_ENDPOINT", "rag-handson-endpoint", "共通の AI Search エンドポイント")
dbutils.widgets.text("PARTICIPANTS", "users", "権限を付けるグループ名・メールアドレス（カンマ区切り）")

AI_SEARCH_ENDPOINT = dbutils.widgets.get("AI_SEARCH_ENDPOINT").strip()
PARTICIPANTS = [p.strip() for p in dbutils.widgets.get("PARTICIPANTS").split(",") if p.strip()]

print(f"エンドポイント: {AI_SEARCH_ENDPOINT}")
print(f"権限を付ける相手: {PARTICIPANTS}")

## 0-0-2. Standard のエンドポイントを作る

すでに同じ名前のエンドポイントがあれば、それを使う（作り直さない）。作成には数分かかる。

In [0]:
from databricks.ai_search.client import AISearchClient

vsc = AISearchClient(disable_notice=True)

if vsc.endpoint_exists(AI_SEARCH_ENDPOINT):
    print(f"既存のエンドポイントを使う: {AI_SEARCH_ENDPOINT}")
else:
    print(f"エンドポイントを作る（数分かかる）: {AI_SEARCH_ENDPOINT}")
    vsc.create_endpoint_and_wait(name=AI_SEARCH_ENDPOINT, endpoint_type="STANDARD")

print(vsc.get_endpoint(AI_SEARCH_ENDPOINT)["endpoint_status"])

## 0-0-3. 参加者に「使用可能」（CAN USE）の権限を付ける

エンドポイントの権限は3段階ある。

| 権限 | できること |
|---|---|
| 作成可能（CAN CREATE） | エンドポイントの一覧・参照・作成。**索引は作れない** |
| 使用可能（CAN USE） | 上に加えて、**このエンドポイントに索引を作れる** |
| 管理可能（CAN MANAGE） | 上に加えて、エンドポイントの削除と権限の変更 |

参加者には **使用可能** を付ける。管理可能を付けると、参加者がエンドポイントを削除できてしまう。

権限は、エンドポイントの ID を指定して Permissions API（オブジェクトの種類 `vector-search-endpoints`）で付ける。既存の権限は残したまま追加する。

> 索引を検索するだけなら、エンドポイントの権限は要らない。索引に対する Unity Catalog の `SELECT` で足りる。各参加者は自分のスキーマに索引を作るので、自分の索引は自分で検索できる。

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.iam import AccessControlRequest, PermissionLevel

w = WorkspaceClient()
endpoint_id = w.vector_search_endpoints.get_endpoint(AI_SEARCH_ENDPOINT).id

acl = [
    AccessControlRequest(user_name=p, permission_level=PermissionLevel.CAN_USE)
    if "@" in p
    else AccessControlRequest(group_name=p, permission_level=PermissionLevel.CAN_USE)
    for p in PARTICIPANTS
]
w.permissions.update(request_object_type="vector-search-endpoints", request_object_id=endpoint_id, access_control_list=acl)

# 付いた権限を確かめる
perms = w.permissions.get(request_object_type="vector-search-endpoints", request_object_id=endpoint_id)
display(spark.createDataFrame(
    [
        (a.group_name or a.user_name or a.service_principal_name, p.permission_level.value, bool(p.inherited))
        for a in perms.access_control_list
        for p in a.all_permissions
    ],
    "principal STRING, permission STRING, inherited BOOLEAN",
))

参加者のグループ（または参加者）に `CAN_USE` が付いていることを確かめる。最後に、エンドポイントの名前を参加者に伝える。

### 参考：画面で作る・権限を付ける

コードの代わりに、画面でもエンドポイントを作れる。画像はこのノートブックと同じフォルダーの `images/` にある。

#### UI-1：AI検索を開く

①左サイドバーの「コンピュート」をクリックし、②「AI検索」タブを開いて、③「エンドポイントを作成」をクリックする。

![UI-1](./images/aisearch-01.png)

→ エンドポイントを作成するダイアログが表示される。

#### UI-2：エンドポイントを作る

①「名前」に widget の `AI_SEARCH_ENDPOINT` と同じ名前（例 `rag-handson-endpoint`）を入れ、②タイプで「標準」を選んで、③「確認」をクリックする。

![UI-2](./images/aisearch-02.png)

→ エンドポイントの作成が始まる。「準備完了」になるまで数分かかる。

> **注意**：エンドポイントは稼働している間、課金される。

権限は、「コンピュート」→「AI検索」タブでエンドポイントを開き、右上の「⋮」メニューの権限の編集（英語表示では Edit permissions）から付ける。参加者のグループを選び、「使用可能」（Can use）にする。

## 0-0-4. 参加者の索引の数を確かめる（ハンズオン中）

参加者が 1-1 で索引を作ると、このエンドポイントの索引が増える。上限の 50 に近づいていないかを確かめる。

In [0]:
ep = w.vector_search_endpoints.get_endpoint(AI_SEARCH_ENDPOINT)
print(f"{AI_SEARCH_ENDPOINT}: 索引 {ep.num_indexes} 個（上限 50）")

indexes = [(i.name, i.creator) for i in w.vector_search_indexes.list_indexes(endpoint_name=AI_SEARCH_ENDPOINT)]
if indexes:
    display(spark.createDataFrame(indexes, "index_name STRING, creator STRING"))

> 一覧に出るのは、代表者に見る権限がある索引だけである。上の索引の数（`num_indexes`）は、すべての索引を数える。

## 0-0-5. 後片付け（ハンズオンの後）

**全員が終わってから** エンドポイントを削除する。先に削除すると、参加者の索引が使えなくなる。参加者には、先に自分のスキーマ（索引を含む）を削除してもらう（README の「後片付け」）。

次のセルのコメントを外して実行する。

In [0]:
# vsc.delete_endpoint(AI_SEARCH_ENDPOINT)
print("コメントを外すと、共通のエンドポイントを削除する")

## まとめ

- 共通の AI Search エンドポイントを Standard で作り、参加者に「使用可能」の権限を付けた
- 参加者は **00_データ準備** から始め、1-1 で、このエンドポイントに自分の索引を作る

### 参考リンク
- [AI Search の索引を作る](https://docs.databricks.com/aws/ja/ai-search/create-ai-search)
- [アクセス制御（AI Search エンドポイントの権限）](https://docs.databricks.com/aws/ja/security/auth/access-control/)
- [AI Search の上限](https://docs.databricks.com/aws/ja/ai-search/ai-search/)